# 03 · Predict the star rating as a class (1–5)

**Use case:** the same question as 02, but the product team wants class probabilities — how sure is the model this is a 1-star review? — to route angry customers to a person.

**Model / lane:** relational GNN — this time the GATv2 architecture (`model_key="gatv2_full"`), 5 classes, text embeddings

**Sub-tasks**
1. Define the task as **multiclass** classification on `rating`
2. Train the relational GNN
3. Accuracy and macro-F1 on the test split vs the always-5★ baseline (63% of reviews are 5★)
4. Per-class precision / recall from the confusion matrix
5. Probabilities for one review

In [1]:
import sys, json
sys.path.insert(0, "..")
from _common import connect, get_or_create_project, print_schema, train_or_reuse, credits_used, save_metrics, show

ls = connect()

signed in as team@langsat.ai · tier team · key 'TEST_SDK_2' with 17 scopes


In [2]:
p = get_or_create_project(ls, "rating-classes", kind="data_science")
before = credits_used(ls)

reusing project c40a4efc-7d71-4534-85f1-f80135e71ab5 (amazon-reviews-rating-classes, status=ready)
project ready: status=ready · type=data_science · files=['customer.csv', 'product.csv', 'review.csv']


In [3]:
model = train_or_reuse(p, "Classify each review into its star rating from 1 to 5 using the review text, the summary and the product",
                       task_type="supervised", subtask_type="multiclass_classification", enable_text_embedding=True, model_key="gatv2_full")
metrics = model["metrics"]
show(metrics, keys=("acc", "f1_macro", "precision_macro", "recall_macro"))

reusing model cb1c7b94-613b-413a-9bf7-50342f63a953 (GAT, trained 2026-09-15T09:02:45.456967+00:00)
  acc                    0.665
  f1_macro               0.4655
  recall_macro           0.4516
  precision_macro        0.4991


In [4]:
review = p.table("review").to_pandas()
majority = review["rating"].value_counts(normalize=True).max()
print(f"always-5★ baseline · accuracy {majority:.3f} · macro-F1 {(2*majority*1)/(majority+1)/5:.3f} (only one class ever right)")
print(f"model (test split)  · accuracy {metrics.get('acc'):.3f} · macro-F1 {metrics.get('f1_macro'):.3f}")

always-5★ baseline · accuracy 0.631 · macro-F1 0.155 (only one class ever right)
model (test split)  · accuracy 0.665 · macro-F1 0.466


In [5]:
import pandas as pd
cm = metrics.get("confusion_matrix"); labels = metrics.get("class_labels")
if cm and labels:
    df = pd.DataFrame(cm, index=[f"true {l}" for l in labels], columns=[f"pred {l}" for l in labels])
    display(df)
    recall = df.values.diagonal() / df.sum(axis=1).values.clip(min=1)
    print("recall per class:", dict(zip(labels, recall.round(2))))

## The reviews most likely to be 1★

A multiclass model carries five probabilities per row, so a ranking needs to say which class: `class_index=0` is the first label (1★). That list is the routing queue for the support team.

In [6]:
labels = metrics.get("class_labels") or [1, 2, 3, 4, 5]
top = ls.predict.top(model["model_id"], n=5, class_index=0)             # rank by P(rating == labels[0])
print("ranked by:", top.get("ranked_by"), "· class", labels[0], "★ ·", top.get("total_ranked"), "ranked")
for r in top["ranking"]:
    print(f"  #{r['rank']} review {r['entity_id']} · P(1★) = {r.get('score')}")
one = ls.predict.predict(model_id=model["model_id"], entity_id=top["ranking"][0]["entity_id"])
print("\nprediction:", one["result"]["prediction"])
print("probabilities:", {k: round(v, 3) for k, v in (one["result"].get("probabilities") or {}).items()})

ranked by: class_probability · class 1 ★ · 2000 ranked
  #1 review 8451 · P(1★) = 0.9998900890350342
  #2 review 8248 · P(1★) = 0.999875545501709
  #3 review 122 · P(1★) = 0.9998308420181274
  #4 review 133 · P(1★) = 0.9998145699501038
  #5 review 469 · P(1★) = 0.9996777772903442



prediction: 1
probabilities: {'1': 1.0, '2': 0.0, '3': 0.0, '4': 0.0, '5': 0.0}


In [7]:
charged = credits_used(ls) - before
save_metrics(".", {"notebook": "03_rating_classification", "task": "multiclass · review.rating (5 classes)", "model": model.get("model_type"),
                   "project_id": p.id, "model_id": model["model_id"], "training_duration_sec": model.get("training_duration_sec"),
                   "credits_charged_this_run": charged,
                   "headline": {"accuracy": metrics.get("acc"), "f1_macro": metrics.get("f1_macro")},
                   "baseline": {"accuracy": round(float(majority), 4), "f1_macro_note": "always-5★"}})

wrote results/metrics.json


PosixPath('results/metrics.json')